# Pipeline de Detecção de Crises Epilépticas — Notebook 2
## Treinamento e avaliação (LOSO)

Este notebook executa o treinamento e a avaliação em validação **leave-one-subject-out** (LOSO) para comparar o baseline com o pipeline V2.
As métricas são calculadas **por janela** e **por evento** e os resultados são salvos em CSV para análise posterior.

---

## 1. Imports

Carrega bibliotecas de ML, métricas e utilitários para o pipeline (SVM/RF/XGBoost, seleção de features, métricas e progresso).

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    confusion_matrix, roc_auc_score, average_precision_score
)
from xgboost import XGBClassifier
from tqdm.auto import tqdm
import itertools

warnings.filterwarnings('ignore')
print("✅ Imports OK")

✅ Imports OK


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carregar metadados do Notebook 1

Lê o arquivo `pipeline_meta.json` com sujeitos válidos, configurações de janela, frequência de amostragem e índices das features extraídas no Notebook 1.

In [2]:
# Ajustar caminho se necessário
DATA_DIR = './data'
meta_path = os.path.join(DATA_DIR, 'pipeline_meta.json')

with open(meta_path, 'r') as f:
    meta = json.load(f)

VALID_SUBJECTS = meta['VALID_SUBJECTS']
WINDOW_CONFIGS = meta['WINDOW_CONFIGS']
SFREQ = meta['SFREQ']
RANDOM_SEED = 42

# Índices de features
level3_fsa = meta['level3_fsa']

print(f"✅ Metadados carregados")
print(f"   Sujeitos válidos: {VALID_SUBJECTS}")
print(f"   Configurações de janela: {list(WINDOW_CONFIGS.keys())}")

✅ Metadados carregados
   Sujeitos válidos: ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011', 'sub-012']
   Configurações de janela: ['4s_50', '5s_50', '6s_50']


## 3. Configuração dos Experimentos

**54 combinações = 3 janelas × 3 FS × 2 ratios × 3 modelos**

### Feature sets
- **FS-A** — 10 features/canal (5 bandas × [L1-norm, Entropia] via GWST)
- **FS-B** — FS-A + 20 adicionais/canal (7 temp + 11 PSD + 2 entropia) = 30 total/canal
- **FS-C** — 34 features/canal (30 DWT + 3 Hjorth + 1 Line Length)

### Ratios de undersampling (aplicados apenas no treino)
- **1:3** — 3 non-seizure para cada seizure
- **1:5** — 5 non-seizure para cada seizure

In [3]:
# Ajustar caminho se necessário
DATA_DIR = './data'
meta_path = os.path.join(DATA_DIR, 'pipeline_meta.json')

with open(meta_path, 'r') as f:
    meta = json.load(f)

VALID_SUBJECTS = meta['VALID_SUBJECTS']
WINDOW_CONFIGS = meta['WINDOW_CONFIGS']
SFREQ = meta['SFREQ']
RANDOM_SEED = 42

# Índices de features (TODAS as feature sets)
level3_fsa = meta['level3_fsa']
level3_fsb = meta['level3_fsb']
level3_fsc = meta['level3_fsc']

print(f"✅ Metadados carregados")
print(f"   Sujeitos válidos: {VALID_SUBJECTS}")
print(f"   Configurações de janela: {list(WINDOW_CONFIGS.keys())}")

✅ Metadados carregados
   Sujeitos válidos: ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011', 'sub-012']
   Configurações de janela: ['4s_50', '5s_50', '6s_50']


## 4. Funções de undersampling e métricas

Define o undersampling simples do baseline e as métricas **por janela** e **por evento** (sensibilidade, precisão e FAR/hora).
As funções de evento convertem janelas em eventos contínuos para medir detecção clínica.

In [4]:
def apply_ratio(X, y, ratio, seed=RANDOM_SEED):
    """
    Reamostra para ratio non-seizure : seizure.
    Mantem todas as seizure, faz STRIDE nas non-seizure.
    """
    idx_sz  = np.where(y == 1)[0]
    idx_non = np.where(y == 0)[0]
    
    if len(idx_sz) == 0:
        return X[:0], y[:0]
    
    n_want = min(len(idx_non), ratio * len(idx_sz))
    stride = max(1, len(idx_non) // n_want)
    idx_sel = idx_non[::stride][:n_want]
    
    idx_all = np.sort(np.concatenate([idx_sz, idx_sel]))
    return X[idx_all], y[idx_all]


def compute_far(y_test, y_pred, win_sec, overlap):
    """False Alarm Rate por hora (baseado em janelas)."""
    step_sec = win_sec * (1.0 - overlap)
    total_hours = (len(y_test) * step_sec) / 3600.0
    if total_hours == 0:
        return 0.0
    false_positives = ((y_pred == 1) & (y_test == 0)).sum()
    return false_positives / total_hours


def compute_metrics(y_test, y_pred, win_sec, overlap):
    """Calcula metricas por janela."""
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total > 0 else 0.0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = f1_score(y_test, y_pred, zero_division=0)
    far = compute_far(y_test, y_pred, win_sec, overlap)
    
    return {
        'accuracy': accuracy,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'f1': f1,
        'far_per_hour': far,
        'tp': int(tp),
        'fp': int(fp),
        'tn': int(tn),
        'fn': int(fn)
    }


def windows_to_events(y, win_sec, overlap):
    """Converte vetor binario em lista de eventos (start_time, end_time)."""
    step_sec = win_sec * (1.0 - overlap)
    events = []
    in_event = False
    event_start = 0.0
    
    for i, val in enumerate(y):
        if val == 1 and not in_event:
            event_start = i * step_sec
            in_event = True
        elif val == 0 and in_event:
            event_end = i * step_sec
            events.append((event_start, event_end))
            in_event = False
    
    if in_event:
        event_end = len(y) * step_sec
        events.append((event_start, event_end))
    
    return events


def compute_event_metrics(y_true, y_pred, win_sec, overlap):
    """Calcula metricas por evento a partir de vetores por janela."""
    true_events = windows_to_events(y_true, win_sec, overlap)
    pred_events = windows_to_events(y_pred, win_sec, overlap)
    step_sec = win_sec * (1.0 - overlap)
    total_hours = (len(y_true) * step_sec) / 3600.0
    if total_hours == 0:
        total_hours = 0.0
    
    matched_true = set()
    matched_pred = 0
    
    for pe in pred_events:
        p_start, p_end = pe
        hit = False
        for idx, te in enumerate(true_events):
            t_start, t_end = te
            overlap_sec = min(p_end, t_end) - max(p_start, t_start)
            if overlap_sec > 0:
                matched_true.add(idx)
                hit = True
                break
        if hit:
            matched_pred += 1
    
    event_sensitivity = (len(matched_true) / len(true_events)) if len(true_events) > 0 else 0.0
    event_precision = (matched_pred / len(pred_events)) if len(pred_events) > 0 else 0.0
    false_alarms = len(pred_events) - matched_pred
    event_far = (false_alarms / total_hours) if total_hours > 0 else 0.0
    
    return {
        'event_sensitivity': event_sensitivity,
        'event_precision': event_precision,
        'event_far_per_hour': event_far
    }

print("✅ Funcoes auxiliares definidas")

✅ Funcoes auxiliares definidas


## 5. Função LOSO genérica (baseline)

Executa o ciclo LOSO para o baseline: carrega dados, aplica undersampling **apenas no treino**, normaliza, seleciona features, treina o modelo e calcula métricas por janela e por evento.

In [5]:
def run_loso(feat_index, subjects, win_sec, overlap, ratio, model_key):
    """
    Leave-One-Subject-Out cross-validation.
    
    Args:
        feat_index: dict {subject: (feat_path, labels_path)}
        subjects: lista de sujeitos validos
        win_sec: tamanho da janela em segundos
        overlap: fracao de overlap (0.5 = 50%)
        ratio: ratio non-seizure:seizure (ex: 3 ou 5) — APLICADO APENAS NO TREINO
        model_key: 'xgb', 'svm', ou 'rf'
    
    Returns:
        Lista de dicionarios com metricas por sujeito.
    """
    valid = [s for s in subjects if s in feat_index]
    results = []
    
    for test_sub in valid:
        train_subs = [s for s in valid if s != test_sub]
        
        # Carregar treino
        X_train_list, y_train_list = [], []
        for s in train_subs:
            fp, lp = feat_index[s]
            X_train_list.append(np.load(fp))
            y_train_list.append(np.load(lp))
        
        X_train = np.vstack(X_train_list)
        y_train = np.concatenate(y_train_list)
        
        # Undersampling no treino
        X_train, y_train = apply_ratio(X_train, y_train, ratio)
        
        # Carregar teste (dados inteiros, sem ratio)
        fp_test, lp_test = feat_index[test_sub]
        X_test = np.load(fp_test)
        y_test = np.load(lp_test)
        
        if len(X_train) == 0 or len(X_test) == 0:
            continue
        
        # Normalizacao
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        # Feature selection
        n_features = min(10, X_train.shape[1])
        selector = SelectKBest(mutual_info_classif, k=n_features)
        X_train = selector.fit_transform(X_train, y_train)
        X_test = selector.transform(X_test)
        
        # Modelo
        if model_key == 'xgb':
            model = XGBClassifier(
                n_estimators=100,
                max_depth=5,
                learning_rate=0.1,
                random_state=RANDOM_SEED,
                eval_metric='logloss'
            )
        elif model_key == 'svm':
            model = SVC(
                kernel='rbf',
                C=1.0,
                gamma='scale',
                probability=True,
                random_state=RANDOM_SEED
            )
        elif model_key == 'rf':
            model = RandomForestClassifier(
                n_estimators=100,
                max_depth=10,
                random_state=RANDOM_SEED
            )
        
        model.fit(X_train, y_train)
        
        if hasattr(model, 'predict_proba'):
            probs = model.predict_proba(X_test)[:, 1]
        elif hasattr(model, 'decision_function'):
            scores = model.decision_function(X_test)
            if scores.max() != scores.min():
                probs = (scores - scores.min()) / (scores.max() - scores.min())
            else:
                probs = np.zeros_like(scores)
        else:
            probs = model.predict(X_test)
        
        if np.issubdtype(probs.dtype, np.floating):
            y_pred = (probs >= 0.5).astype(int)
        else:
            y_pred = probs.astype(int)
        
        # Metricas
        metrics = compute_metrics(y_test, y_pred, win_sec, overlap)
        metrics.update(compute_event_metrics(y_test, y_pred, win_sec, overlap))
        
        if np.issubdtype(probs.dtype, np.floating) and len(np.unique(y_test)) > 1:
            metrics['auc_pr'] = average_precision_score(y_test, probs)
        else:
            metrics['auc_pr'] = 0.0
        
        metrics['subject'] = test_sub
        results.append(metrics)
    
    return results

print("✅ Funcao LOSO definida")

✅ Funcao LOSO definida


## 6. Executar os experimentos (baseline)

Nos próximos blocos o notebook define utilitários opcionais, configura as combinações e executa o loop baseline para todas as combinações de FS × janela × ratio × modelo.

## 6.0. Diagnóstico opcional de overfitting/underfitting

Define funções para calcular e visualizar learning curves. **Não é usado no loop principal**, mas pode ser chamado depois para investigar estabilidade do treinamento.

In [6]:
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

def compute_learning_curve(model, X_train, y_train, cv=5):
    """
    Calcula learning curve para detectar overfitting/underfitting.
    Retorna train_sizes, train_scores, val_scores.
    """
    train_sizes = np.linspace(0.1, 1.0, 10)
    
    train_sizes_abs, train_scores, val_scores = learning_curve(
        model, X_train, y_train,
        train_sizes=train_sizes,
        cv=cv,
        scoring='f1',
        n_jobs=-1,
        random_state=RANDOM_SEED
    )
    
    return train_sizes_abs, train_scores, val_scores


def diagnose_fitting(train_scores, val_scores):
    """
    Diagnóstico automático de overfitting/underfitting.
    
    Returns:
        - 'overfitting': train >> val (gap > 0.15)
        - 'underfitting': both low (< 0.5)
        - 'good_fit': train ≈ val e ambos razoáveis
    """
    train_mean = np.mean(train_scores[-1])  # Última época
    val_mean = np.mean(val_scores[-1])
    gap = train_mean - val_mean
    
    if train_mean < 0.5 and val_mean < 0.5:
        return 'underfitting', gap
    elif gap > 0.15:
        return 'overfitting', gap
    else:
        return 'good_fit', gap


def plot_learning_curve(train_sizes, train_scores, val_scores, title):
    """
    Plota learning curve (train vs validation).
    """
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    
    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes, train_mean, 'o-', color='r', label='Treino')
    plt.plot(train_sizes, val_mean, 'o-', color='g', label='Validação')
    
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='r')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='g')
    
    plt.xlabel('Número de amostras de treino')
    plt.ylabel('F1-Score')
    plt.title(title)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    return plt


def save_training_diagnostics(train_sizes, train_scores, val_scores, config_name, fs_name):
    """
    Salva dados de diagnóstico em CSV para análise posterior.
    """
    diagnosis, gap = diagnose_fitting(train_scores, val_scores)
    
    # Criar DataFrame
    data = {
        'train_size': train_sizes,
        'train_mean': np.mean(train_scores, axis=1),
        'train_std': np.std(train_scores, axis=1),
        'val_mean': np.mean(val_scores, axis=1),
        'val_std': np.std(val_scores, axis=1),
    }
    df = pd.DataFrame(data)
    
    # Adicionar metadados
    df['config'] = config_name
    df['fs'] = fs_name
    df['diagnosis'] = diagnosis
    df['gap'] = gap
    
    return df, diagnosis, gap


print("✅ Funções de diagnóstico definidas")

✅ Funções de diagnóstico definidas


## 6.0.2. Checagem rápida (baseline)

Compara **sensibilidade** no treino vs teste para uma única configuração do baseline.
Inclui sensibilidade por evento como referência clínica.

In [7]:
# Checagem rapida (baseline): uma configuracao e um sujeito de teste
if 'RUN_CFGS' not in globals():
    RUN_CFGS = list(WINDOW_CONFIGS.keys())
if 'FS_INDICES' not in globals():
    FS_INDICES = {'FS-A': level3_fsa}

fs_name = 'FS-A'
cfg_name = RUN_CFGS[0]
ratio = 3
model_key = 'svm'

cfg = WINDOW_CONFIGS[cfg_name]
feat_index = FS_INDICES[fs_name].get(cfg_name, {})
valid_subs = [s for s in VALID_SUBJECTS if s in feat_index]

if len(valid_subs) < 2:
    print("⚠️ Poucos sujeitos para checagem rapida.")
else:
    test_sub = valid_subs[0]
    train_subs = [s for s in valid_subs if s != test_sub]

    X_train_list, y_train_list = [], []
    for s in tqdm(train_subs, desc='Carregando treino (baseline)', leave=True):
        fp, lp = feat_index[s]
        X_train_list.append(np.load(fp))
        y_train_list.append(np.load(lp))

    X_train = np.vstack(X_train_list)
    y_train = np.concatenate(y_train_list)
    X_train, y_train = apply_ratio(X_train, y_train, ratio)

    fp_test, lp_test = feat_index[test_sub]
    X_test = np.load(fp_test)
    y_test = np.load(lp_test)

    if len(X_train) == 0 or len(X_test) == 0:
        print("⚠️ Sem dados suficientes para a checagem rapida.")
    else:
        max_train = 20000
        if len(X_train) > max_train:
            idx = np.random.choice(len(X_train), size=max_train, replace=False)
            X_train = X_train[idx]
            y_train = y_train[idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        n_features = min(10, X_train.shape[1])
        selector = SelectKBest(mutual_info_classif, k=n_features)
        X_train = selector.fit_transform(X_train, y_train)
        X_test = selector.transform(X_test)

        if model_key == 'xgb':
            model = XGBClassifier(
                n_estimators=100,
                max_depth=5,
                learning_rate=0.1,
                random_state=RANDOM_SEED,
                eval_metric='logloss'
            )
        elif model_key == 'svm':
            model = SVC(
                kernel='rbf',
                C=1.0,
                gamma='scale',
                probability=True,
                random_state=RANDOM_SEED
            )
        else:
            model = RandomForestClassifier(
                n_estimators=100,
                max_depth=10,
                random_state=RANDOM_SEED
            )

        model.fit(X_train, y_train)
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        train_metrics = compute_metrics(y_train, y_train_pred, cfg['win_sec'], cfg['overlap'])
        test_metrics = compute_metrics(y_test, y_test_pred, cfg['win_sec'], cfg['overlap'])
        train_events = compute_event_metrics(y_train, y_train_pred, cfg['win_sec'], cfg['overlap'])
        test_events = compute_event_metrics(y_test, y_test_pred, cfg['win_sec'], cfg['overlap'])

        train_sens = train_metrics['sensitivity']
        test_sens = test_metrics['sensitivity']
        gap = train_sens - test_sens

        if train_sens < 0.5 and test_sens < 0.5:
            diagnosis = 'underfitting'
        elif gap > 0.15:
            diagnosis = 'overfitting'
        else:
            diagnosis = 'good_fit'

        print("\n=== CHECAGEM RAPIDA (BASELINE) ===")
        print(f"FS: {fs_name} | Janela: {cfg_name} | Modelo: {model_key.upper()} | Ratio: 1:{ratio}")
        print(f"Sensibilidade treino: {train_sens:.3f}")
        print(f"Sensibilidade teste:  {test_sens:.3f}")
        print(f"Gap:                 {gap:+.3f}")
        print(f"Event sens treino:   {train_events['event_sensitivity']:.3f}")
        print(f"Event sens teste:    {test_events['event_sensitivity']:.3f}")
        print(f"Diagnostico: {diagnosis}")

Carregando treino (baseline): 100%|██████████| 11/11 [00:00<00:00, 24.89it/s]



=== CHECAGEM RAPIDA (BASELINE) ===
FS: FS-A | Janela: 4s_50 | Modelo: SVM | Ratio: 1:3
Sensibilidade treino: 0.653
Sensibilidade teste:  0.556
Gap:                 +0.098
Event sens treino:   0.944
Event sens teste:    1.000
Diagnostico: good_fit


## 6.0.1. Configuração do experimento e estruturas de resultado

Cria listas de combinações (janelas, modelos, ratios) e o dicionário `full_results` que receberá os resultados do baseline.

In [8]:
# Configuracoes do experimento
RUN_CFGS = list(WINDOW_CONFIGS.keys())  # ['4s_50', '5s_50', '6s_50']
RUN_MODELS = ['xgb', 'svm', 'rf']       # XGBoost, SVM, Random Forest
UNDERSAMPLE_RATIOS = [3, 5]             # 1:3 e 1:5

# Mapeamento FS -> indices
FS_INDICES = {
    'FS-A': level3_fsa,
    'FS-B': level3_fsb,
    'FS-C': level3_fsc,
}

# Estrutura para armazenar resultados
full_results = {fs: {} for fs in FS_INDICES.keys()}

print("✅ Configuracoes definidas")
print(f"   FS: {list(FS_INDICES.keys())}")
print(f"   RUN_CFGS: {RUN_CFGS}")
print(f"   UNDERSAMPLE_RATIOS: {UNDERSAMPLE_RATIOS}")
print(f"   RUN_MODELS: {RUN_MODELS}")

✅ Configuracoes definidas
   FS: ['FS-A', 'FS-B', 'FS-C']
   RUN_CFGS: ['4s_50', '5s_50', '6s_50']
   UNDERSAMPLE_RATIOS: [3, 5]
   RUN_MODELS: ['xgb', 'svm', 'rf']


In [9]:
# Gerar todas as combinações
combos = list(itertools.product(
    FS_INDICES.keys(),
    RUN_CFGS,
    UNDERSAMPLE_RATIOS,
    RUN_MODELS
))

print(f"\nTotal de experimentos: {len(combos)}")
print("Iniciando loop principal...\n")

for fs_name, cfg_name, ratio, model_key in tqdm(combos, desc='Experimentos', unit='exp'):
    cfg = WINDOW_CONFIGS[cfg_name]
    feat_index = FS_INDICES[fs_name].get(cfg_name, {})
    valid_subs = [s for s in VALID_SUBJECTS if s in feat_index]
    
    if not valid_subs:
        continue
    
    # Criar estrutura de dicionário
    if cfg_name not in full_results[fs_name]:
        full_results[fs_name][cfg_name] = {}
    if ratio not in full_results[fs_name][cfg_name]:
        full_results[fs_name][cfg_name][ratio] = {}
    
    # Executar LOSO
    results = run_loso(
        feat_index=feat_index,
        subjects=valid_subs,
        win_sec=cfg['win_sec'],
        overlap=cfg['overlap'],
        ratio=ratio,
        model_key=model_key
    )
    
    # Converter para DataFrame
    df = pd.DataFrame(results)
    full_results[fs_name][cfg_name][ratio][model_key] = (results, df)

print("\n✅ Todos os experimentos baseline concluídos!")


Total de experimentos: 54
Iniciando loop principal...



Experimentos: 100%|██████████| 54/54 [1:12:17<00:00, 80.32s/exp] 


✅ Todos os experimentos baseline concluídos!


## 6.1. Resumo de métricas (baseline)

Agrega as métricas por configuração após o loop baseline, gerando uma visão rápida das melhores combinações.

In [10]:
# Resumo agregado por janela (baseline)
metrics_window = [
    'accuracy', 'f1', 'sensitivity', 'specificity', 'precision',
    'auc_pr', 'far_per_hour', 'event_sensitivity', 'event_precision', 'event_far_per_hour'
 ]
baseline_rows = []

for fs_name in FS_INDICES.keys():
    for cfg_name in RUN_CFGS:
        for ratio in UNDERSAMPLE_RATIOS:
            for model_key in RUN_MODELS:
                try:
                    _, df_baseline = full_results[fs_name][cfg_name][ratio][model_key]
                    row = {
                        'FS': fs_name,
                        'Janela': cfg_name,
                        'Ratio': f"1:{ratio}",
                        'Modelo': model_key.upper(),
                    }
                    for m in metrics_window:
                        row[m] = df_baseline[m].mean() if m in df_baseline else 0.0
                    baseline_rows.append(row)
                except KeyError:
                    pass

df_baseline_window_summary = pd.DataFrame(baseline_rows)
print(f"Resumo baseline (janela) gerado: {len(df_baseline_window_summary)} linhas")

df_baseline_window_summary = df_baseline_window_summary.sort_values(
    by=['sensitivity', 'far_per_hour', 'auc_pr'],
    ascending=[False, True, False]
).reset_index(drop=True)

display(df_baseline_window_summary.head(10))

Resumo baseline (janela) gerado: 54 linhas


,FS,Janela,Ratio,Modelo,accuracy,f1,sensitivity,specificity,precision,auc_pr,far_per_hour,event_sensitivity,event_precision,event_far_per_hour
0,FS-B,4s_50,1:3,RF,0.903247,0.062359,0.609374,0.904418,0.034256,0.154989,171.166518,0.854167,0.011602,61.343936
1,FS-B,4s_50,1:3,XGB,0.905825,0.061300,0.600211,0.907071,0.033546,0.130991,166.424693,0.854167,0.012336,61.559055
2,FS-A,4s_50,1:3,XGB,0.911284,0.059813,0.577975,0.912697,0.032698,0.129642,156.366943,0.854167,0.011300,62.158336
3,FS-A,5s_50,1:3,SVM,0.920483,0.058268,0.575742,0.921955,0.031538,0.168958,111.833119,0.854167,0.011175,38.112932
4,FS-A,5s_50,1:3,RF,0.917092,0.061142,0.573872,0.918530,0.033711,0.138782,116.724207,0.854167,0.012647,44.244462
5,FS-A,4s_50,1:3,RF,0.913217,0.058820,0.572955,0.914631,0.032069,0.133395,152.874532,0.854167,0.011310,56.537704
6,FS-C,5s_50,1:3,XGB,0.906205,0.055090,0.570912,0.907584,0.029923,0.153612,132.388342,0.937500,0.010823,48.980914
7,FS-B,6s_50,1:3,RF,0.913049,0.072744,0.569242,0.914422,0.041337,0.145426,102.149407,0.854167,0.012737,39.945703
8,FS-B,6s_50,1:3,XGB,0.913484,0.070753,0.568282,0.914899,0.040035,0.134702,101.589894,0.848611,0.012996,40.462588
9,FS-C,4s_50,1:3,SVM,0.907346,0.048357,0.567397,0.908753,0.025628,0.124120,163.439006,0.854167,0.010294,50.338811


## 7. Salvar resultados consolidados (baseline)

Consolida todas as execuções LOSO do baseline em um único CSV para análise externa.

In [12]:
# Consolidar tudo em um unico CSV
all_rows = []

for fs_name in FS_INDICES.keys():
    for cfg_name in RUN_CFGS:
        for ratio in UNDERSAMPLE_RATIOS:
            for model_key in RUN_MODELS:
                try:
                    results, df = full_results[fs_name][cfg_name][ratio][model_key]
                    for _, row in df.iterrows():
                        row_dict = row.to_dict()
                        row_dict['fs_name'] = fs_name
                        row_dict['cfg_name'] = cfg_name
                        row_dict['ratio'] = ratio
                        row_dict['model'] = model_key
                        all_rows.append(row_dict)
                except KeyError:
                    pass

df_all_baseline = pd.DataFrame(all_rows)
output_path = os.path.join(DATA_DIR, 'results_baseline.csv')
df_all_baseline.to_csv(output_path, index=False)

print(f"✅ Resultados baseline salvos em: {output_path}")
print(f"   Total de linhas: {len(df_all_baseline)}")

✅ Resultados baseline salvos em: ./data\results_baseline.csv
   Total de linhas: 648


## 8. Ranking final (baseline)

Ordena as configurações pela média por sujeito, priorizando **sensibilidade alta**, **FAR baixo** e **AUC-PR**.

In [13]:
# Agregar por configuracao (media entre sujeitos)
metrics = [
    'accuracy', 'f1', 'sensitivity', 'specificity', 'precision',
    'auc_pr', 'far_per_hour', 'event_sensitivity', 'event_precision', 'event_far_per_hour'
 ]
ranking_rows = []

for fs_name in FS_INDICES.keys():
    for cfg_name in RUN_CFGS:
        for ratio in UNDERSAMPLE_RATIOS:
            for model_key in RUN_MODELS:
                try:
                    _, df = full_results[fs_name][cfg_name][ratio][model_key]
                    row = {
                        'FS': fs_name,
                        'Janela': cfg_name,
                        'Ratio': f"1:{ratio}",
                        'Modelo': model_key.upper(),
                    }
                    for m in metrics:
                        row[m] = df[m].mean()
                    ranking_rows.append(row)
                except KeyError:
                    pass

df_ranking_baseline = pd.DataFrame(ranking_rows)
df_ranking_baseline = df_ranking_baseline.sort_values(
    by=['sensitivity', 'far_per_hour', 'auc_pr'],
    ascending=[False, True, False]
).reset_index(drop=True)

ranking_path = os.path.join(DATA_DIR, 'ranking_baseline.csv')
df_ranking_baseline.to_csv(ranking_path, index=False)

print("✅ Ranking baseline salvo")
print("\nTop 10 configuracoes (baseline):")
display(df_ranking_baseline.head(10))

✅ Ranking baseline salvo

Top 10 configuracoes (baseline):


,FS,Janela,Ratio,Modelo,accuracy,f1,sensitivity,specificity,precision,auc_pr,far_per_hour,event_sensitivity,event_precision,event_far_per_hour
0,FS-B,4s_50,1:3,RF,0.903247,0.062359,0.609374,0.904418,0.034256,0.154989,171.166518,0.854167,0.011602,61.343936
1,FS-B,4s_50,1:3,XGB,0.905825,0.061300,0.600211,0.907071,0.033546,0.130991,166.424693,0.854167,0.012336,61.559055
2,FS-A,4s_50,1:3,XGB,0.911284,0.059813,0.577975,0.912697,0.032698,0.129642,156.366943,0.854167,0.011300,62.158336
3,FS-A,5s_50,1:3,SVM,0.920483,0.058268,0.575742,0.921955,0.031538,0.168958,111.833119,0.854167,0.011175,38.112932
4,FS-A,5s_50,1:3,RF,0.917092,0.061142,0.573872,0.918530,0.033711,0.138782,116.724207,0.854167,0.012647,44.244462
5,FS-A,4s_50,1:3,RF,0.913217,0.058820,0.572955,0.914631,0.032069,0.133395,152.874532,0.854167,0.011310,56.537704
6,FS-C,5s_50,1:3,XGB,0.906205,0.055090,0.570912,0.907584,0.029923,0.153612,132.388342,0.937500,0.010823,48.980914
7,FS-B,6s_50,1:3,RF,0.913049,0.072744,0.569242,0.914422,0.041337,0.145426,102.149407,0.854167,0.012737,39.945703
8,FS-B,6s_50,1:3,XGB,0.913484,0.070753,0.568282,0.914899,0.040035,0.134702,101.589894,0.848611,0.012996,40.462588
9,FS-C,4s_50,1:3,SVM,0.907346,0.048357,0.567397,0.908753,0.025628,0.124120,163.439006,0.854167,0.010294,50.338811


---

# ========================================
# PARTE 2 - PIPELINE MELHORADO (V2)
# ========================================

## Objetivo
Aprimorar o pipeline com pós-processamento temporal e avaliação por eventos, mantendo a execução **rápida** na Fase 1 e deixando o tuning detalhado para a Fase 2.

## Resumo do que muda na V2
- **Undersampling temporal (hard negatives)** no treino.
- **Probabilidades** no teste + pós-processamento (histerese, k consecutivas, merge e duração mínima).
- **Métricas por janela e por evento** para avaliar viabilidade clínica.
- **Fase 1:** parâmetros fixos (`PARAMS_V2`) para comparar modelos/FS/janelas rapidamente.
- **Fase 2:** tuning apenas do **modelo campeão**.

---

## 9. Undersampling melhorado (V2)

Define o hard negative mining temporal: mantém janelas ictais e seleciona janelas não-ictais próximas no tempo para tornar o treino mais discriminativo.

In [14]:
def apply_ratio_v2(X, y, ratio, win_sec, overlap, seed=RANDOM_SEED):
    """
    Hard negative mining temporal.
    
    Para cada evento de crise:
    - Mantém todas as janelas ictais
    - Seleciona janelas não-ictais em janela temporal de ±60s ao redor
    - Completa com amostras aleatórias se necessário
    
    Args:
        X: features [n_windows, n_features]
        y: labels [n_windows]
        ratio: quantas non-seizure para cada seizure
        win_sec: tamanho da janela em segundos
        overlap: fração de overlap (0.5 = 50%)
        seed: random seed
    
    Returns:
        X_sampled, y_sampled: arrays reamostrados
    """
    np.random.seed(seed)
    
    idx_sz = np.where(y == 1)[0]
    idx_non = np.where(y == 0)[0]
    
    if len(idx_sz) == 0:
        return X[:0], y[:0]
    
    # Passo 1: Identificar eventos de crise (sequências contínuas)
    seizure_events = []
    current_event = [idx_sz[0]]
    
    for i in range(1, len(idx_sz)):
        if idx_sz[i] == idx_sz[i-1] + 1:
            current_event.append(idx_sz[i])
        else:
            seizure_events.append(current_event)
            current_event = [idx_sz[i]]
    seizure_events.append(current_event)
    
    # Passo 2: Para cada evento, selecionar hard negatives
    step_sec = win_sec * (1.0 - overlap)
    context_window_sec = 60.0  # ±60s ao redor da crise
    context_window_idx = int(context_window_sec / step_sec)
    
    hard_negatives = set()
    
    for event_indices in seizure_events:
        event_start = event_indices[0]
        event_end = event_indices[-1]
        
        # Janela antes da crise
        before_start = max(0, event_start - context_window_idx)
        before_end = event_start
        
        # Janela depois da crise
        after_start = event_end + 1
        after_end = min(len(y), event_end + context_window_idx + 1)
        
        # Selecionar não-ictais na janela temporal
        before_candidates = [i for i in range(before_start, before_end) if y[i] == 0]
        after_candidates = [i for i in range(after_start, after_end) if y[i] == 0]
        
        hard_negatives.update(before_candidates)
        hard_negatives.update(after_candidates)
    
    hard_negatives = np.array(sorted(hard_negatives))
    
    # Passo 3: Completar com amostras aleatórias se necessário
    n_want = ratio * len(idx_sz)
    
    if len(hard_negatives) >= n_want:
        # Temos hard negatives suficientes, selecionar aleatoriamente
        selected_non = np.random.choice(hard_negatives, size=n_want, replace=False)
    else:
        # Usar todos os hard negatives + complementar com aleatórios
        remaining = [i for i in idx_non if i not in hard_negatives]
        n_remaining = n_want - len(hard_negatives)
        
        if len(remaining) >= n_remaining:
            extra = np.random.choice(remaining, size=n_remaining, replace=False)
        else:
            extra = np.array(remaining)
        
        selected_non = np.concatenate([hard_negatives, extra])
    
    # Passo 4: Combinar seizure + non-seizure
    idx_all = np.sort(np.concatenate([idx_sz, selected_non]))
    
    return X[idx_all], y[idx_all]

print("✅ Função apply_ratio_v2 (hard negative mining) definida")

✅ Função apply_ratio_v2 (hard negative mining) definida


## 10. Pós-processamento temporal (V2)

Define histerese, filtro de k consecutivas, conversão janelas→eventos, merge de eventos próximos e remoção de eventos curtos.
Essas etapas reduzem alarmes espúrios e aproximam a avaliação do cenário clínico.

In [16]:
def apply_hysteresis(probs, th_on=0.5, th_off=0.3):
    """
    Aplica histerese em probabilidades.
    
    Estado ON: ativado quando prob >= th_on
    Estado OFF: desativado quando prob < th_off
    Entre th_off e th_on: mantem estado anterior
    """
    detections = np.zeros(len(probs), dtype=int)
    state = 0

    for i, p in enumerate(probs):
        if state == 0:
            if p >= th_on:
                state = 1
        else:
            if p < th_off:
                state = 0
        detections[i] = state

    return detections


def apply_hysteresis_from_pred(probs, detections_init, th_on=0.5, th_off=0.3):
    """
    Histerese inicializada a partir de detections_init.
    """
    detections = np.zeros(len(probs), dtype=int)
    state = int(detections_init[0]) if len(detections_init) > 0 else 0

    for i, p in enumerate(probs):
        if state == 0:
            if p >= th_on:
                state = 1
        else:
            if p < th_off:
                state = 0
        detections[i] = state

    return detections


def apply_k_consecutive(detections, k=3):
    """
    Filtro de k janelas consecutivas.
    Uma janela so e positiva se ela E as (k-1) seguintes forem positivas.
    """
    filtered = np.zeros(len(detections), dtype=int)
    
    for i in range(len(detections) - k + 1):
        if np.all(detections[i:i+k] == 1):
            filtered[i:i+k] = 1
    
    return filtered


def probs_to_events(detections, win_sec, overlap):
    """
    Converte janelas binarias para eventos (start_time, end_time).
    """
    step_sec = win_sec * (1.0 - overlap)
    events = []
    in_event = False
    event_start = 0
    
    for i, det in enumerate(detections):
        if det == 1 and not in_event:
            event_start = i * step_sec
            in_event = True
        elif det == 0 and in_event:
            event_end = i * step_sec
            events.append((event_start, event_end))
            in_event = False
    
    if in_event:
        event_end = len(detections) * step_sec
        events.append((event_start, event_end))
    
    return events


def merge_events(events, max_gap=10.0):
    """
    Une eventos proximos se gap < max_gap segundos.
    """
    if not events:
        return []
    
    events = sorted(events, key=lambda x: x[0])
    merged = [events[0]]
    
    for current in events[1:]:
        last_start, last_end = merged[-1]
        curr_start, curr_end = current
        
        if curr_start - last_end <= max_gap:
            merged[-1] = (last_start, max(last_end, curr_end))
        else:
            merged.append(current)
    
    return merged


def filter_short_events(events, min_duration=5.0):
    """
    Remove eventos com duracao < min_duration.
    """
    return [(s, e) for s, e in events if (e - s) >= min_duration]


def events_to_detections(events, n_windows, win_sec, overlap):
    """
    Converte eventos (segundos) para deteccoes por janela.
    """
    step_sec = win_sec * (1.0 - overlap)
    detections = np.zeros(n_windows, dtype=int)
    
    for i in range(n_windows):
        win_start = i * step_sec
        win_end = win_start + win_sec
        for ev_start, ev_end in events:
            if ev_end > win_start and ev_start < win_end:
                detections[i] = 1
                break
    
    return detections

print("✅ Funcoes de pos-processamento temporal definidas")

✅ Funcoes de pos-processamento temporal definidas


## 11. Métricas usadas na V2

A V2 reutiliza as funções de métricas já definidas:
- **Por janela**: `compute_metrics`
- **Por evento**: `compute_event_metrics`

## 12. Função LOSO V2 (Fase 1)

Executa LOSO com undersampling temporal no treino e pós-processamento fixo no teste.
Retorna métricas por janela e por evento, além dos parâmetros usados.

In [17]:
def run_loso_v2(
    feat_index,
    subjects,
    win_sec,
    overlap,
    model_key,
    ratio_train=3,
    th_on=0.5,
    th_off=0.3,
    k=3,
    max_gap=10.0,
    min_duration=5.0,
 ):
    """
    LOSO com metricas por janela e pos-processamento fixo (Fase 1).
    """
    valid = [s for s in subjects if s in feat_index]
    results = []
    
    for test_sub in valid:
        train_subs = [s for s in valid if s != test_sub]
        
        X_train_list, y_train_list = [], []
        for s in train_subs:
            fp, lp = feat_index[s]
            X_train_list.append(np.load(fp))
            y_train_list.append(np.load(lp))
        
        X_train = np.vstack(X_train_list)
        y_train = np.concatenate(y_train_list)
        
        X_train, y_train = apply_ratio_v2(X_train, y_train, ratio_train, win_sec, overlap)
        
        fp_test, lp_test = feat_index[test_sub]
        X_test = np.load(fp_test)
        y_test = np.load(lp_test)
        
        if len(X_train) == 0 or len(X_test) == 0:
            continue
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        n_features = min(10, X_train.shape[1])
        selector = SelectKBest(mutual_info_classif, k=n_features)
        X_train = selector.fit_transform(X_train, y_train)
        X_test = selector.transform(X_test)
        
        if model_key == 'xgb':
            model = XGBClassifier(
                n_estimators=100,
                max_depth=5,
                learning_rate=0.1,
                random_state=RANDOM_SEED,
                eval_metric='logloss'
            )
        elif model_key == 'svm':
            model = SVC(
                kernel='rbf',
                C=1.0,
                gamma='scale',
                probability=True,
                random_state=RANDOM_SEED
            )
        elif model_key == 'rf':
            model = RandomForestClassifier(
                n_estimators=100,
                max_depth=10,
                random_state=RANDOM_SEED
            )
        
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:, 1]
        auc_pr = average_precision_score(y_test, probs) if len(np.unique(y_test)) > 1 else 0.0
        
        detections_init = (probs >= th_on).astype(int)
        detections = apply_hysteresis_from_pred(probs, detections_init, th_on=th_on, th_off=th_off)
        detections = apply_k_consecutive(detections, k=k)
        
        events = probs_to_events(detections, win_sec, overlap)
        events = merge_events(events, max_gap=max_gap)
        events = filter_short_events(events, min_duration=min_duration)
        detections = events_to_detections(events, len(y_test), win_sec, overlap)
        
        metrics = compute_metrics(y_test, detections, win_sec, overlap)
        metrics.update(compute_event_metrics(y_test, detections, win_sec, overlap))
        metrics['auc_pr'] = auc_pr
        metrics['ratio'] = ratio_train
        metrics['th_on'] = th_on
        metrics['th_off'] = th_off
        metrics['k'] = k
        metrics['max_gap'] = max_gap
        metrics['min_duration'] = min_duration
        metrics['subject'] = test_sub
        results.append(metrics)
    
    return results

print("✅ Funcao run_loso_v2 (fase 1) definida")

✅ Funcao run_loso_v2 (fase 1) definida


## 12.1. Checagem rápida (V2)

Compara **sensibilidade** no treino vs teste para uma única configuração da V2,
já com o pós-processamento temporal aplicado, e mostra sensibilidade por evento.

In [18]:
# Checagem rápida V2: uma configuração e um sujeito de teste
if 'RUN_CFGS' not in globals():
    RUN_CFGS = list(WINDOW_CONFIGS.keys())
if 'FS_INDICES' not in globals():
    FS_INDICES = {'FS-A': level3_fsa}

fs_name = 'FS-A'
cfg_name = RUN_CFGS[0]
ratio_train = 3
model_key = 'svm'
th_on = 0.5
th_off = 0.3
k = 3
max_gap = 10.0
min_duration = 5.0

cfg = WINDOW_CONFIGS[cfg_name]
feat_index = FS_INDICES[fs_name].get(cfg_name, {})
valid_subs = [s for s in VALID_SUBJECTS if s in feat_index]

if len(valid_subs) < 2:
    print("⚠️ Poucos sujeitos para checagem rápida V2.")
else:
    test_sub = valid_subs[0]
    train_subs = [s for s in valid_subs if s != test_sub]

    X_train_list, y_train_list = [], []
    for s in tqdm(train_subs, desc='Carregando treino (V2)', leave=True):
        fp, lp = feat_index[s]
        X_train_list.append(np.load(fp))
        y_train_list.append(np.load(lp))

    X_train = np.vstack(X_train_list)
    y_train = np.concatenate(y_train_list)
    X_train, y_train = apply_ratio_v2(X_train, y_train, ratio_train, cfg['win_sec'], cfg['overlap'])

    fp_test, lp_test = feat_index[test_sub]
    X_test = np.load(fp_test)
    y_test = np.load(lp_test)

    if len(X_train) == 0 or len(X_test) == 0:
        print("⚠️ Sem dados suficientes para a checagem rápida V2.")
    else:
        max_train = 20000
        if len(X_train) > max_train:
            idx = np.random.choice(len(X_train), size=max_train, replace=False)
            X_train = X_train[idx]
            y_train = y_train[idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        n_features = min(10, X_train.shape[1])
        selector = SelectKBest(mutual_info_classif, k=n_features)
        X_train = selector.fit_transform(X_train, y_train)
        X_test = selector.transform(X_test)

        if model_key == 'xgb':
            model = XGBClassifier(
                n_estimators=100,
                max_depth=5,
                learning_rate=0.1,
                random_state=RANDOM_SEED,
                eval_metric='logloss'
            )
        elif model_key == 'svm':
            model = SVC(
                kernel='rbf',
                C=1.0,
                gamma='scale',
                probability=True,
                random_state=RANDOM_SEED
            )
        else:
            model = RandomForestClassifier(
                n_estimators=100,
                max_depth=10,
                random_state=RANDOM_SEED
            )

        model.fit(X_train, y_train)
        probs_train = model.predict_proba(X_train)[:, 1]
        probs_test = model.predict_proba(X_test)[:, 1]

        det_train = apply_hysteresis_from_pred(probs_train, (probs_train >= th_on).astype(int), th_on=th_on, th_off=th_off)
        det_train = apply_k_consecutive(det_train, k=k)
        events_train = probs_to_events(det_train, cfg['win_sec'], cfg['overlap'])
        events_train = merge_events(events_train, max_gap=max_gap)
        events_train = filter_short_events(events_train, min_duration=min_duration)
        det_train = events_to_detections(events_train, len(y_train), cfg['win_sec'], cfg['overlap'])

        det_test = apply_hysteresis_from_pred(probs_test, (probs_test >= th_on).astype(int), th_on=th_on, th_off=th_off)
        det_test = apply_k_consecutive(det_test, k=k)
        events_test = probs_to_events(det_test, cfg['win_sec'], cfg['overlap'])
        events_test = merge_events(events_test, max_gap=max_gap)
        events_test = filter_short_events(events_test, min_duration=min_duration)
        det_test = events_to_detections(events_test, len(y_test), cfg['win_sec'], cfg['overlap'])

        train_metrics = compute_metrics(y_train, det_train, cfg['win_sec'], cfg['overlap'])
        test_metrics = compute_metrics(y_test, det_test, cfg['win_sec'], cfg['overlap'])
        train_events = compute_event_metrics(y_train, det_train, cfg['win_sec'], cfg['overlap'])
        test_events = compute_event_metrics(y_test, det_test, cfg['win_sec'], cfg['overlap'])

        train_sens = train_metrics['sensitivity']
        test_sens = test_metrics['sensitivity']
        gap = train_sens - test_sens

        if train_sens < 0.5 and test_sens < 0.5:
            diagnosis = 'underfitting'
        elif gap > 0.15:
            diagnosis = 'overfitting'
        else:
            diagnosis = 'good_fit'

        print("\n=== CHECAGEM RAPIDA V2 ===")
        print(f"FS: {fs_name} | Janela: {cfg_name} | Modelo: {model_key.upper()} | Ratio: 1:{ratio_train}")
        print(f"th_on: {th_on} | th_off: {th_off} | k: {k} | max_gap: {max_gap} | min_duration: {min_duration}")
        print(f"Sensibilidade treino: {train_sens:.3f}")
        print(f"Sensibilidade teste:  {test_sens:.3f}")
        print(f"Gap:                 {gap:+.3f}")
        print(f"Event sens treino:   {train_events['event_sensitivity']:.3f}")
        print(f"Event sens teste:    {test_events['event_sensitivity']:.3f}")
        print(f"Diagnostico: {diagnosis}")

Carregando treino (V2): 100%|██████████| 11/11 [00:00<00:00, 45.61it/s]



=== CHECAGEM RAPIDA V2 ===
FS: FS-A | Janela: 4s_50 | Modelo: SVM | Ratio: 1:3
th_on: 0.5 | th_off: 0.3 | k: 3 | max_gap: 10.0 | min_duration: 5.0
Sensibilidade treino: 0.642
Sensibilidade teste:  0.599
Gap:                 +0.044
Event sens treino:   0.889
Event sens teste:    1.000
Diagnostico: good_fit


## 13. Executar experimentos V2 (Fase 1)

Roda a V2 com parâmetros fixos (`PARAMS_V2`) para todas as combinações de FS × janela × modelo × ratio.

In [19]:
# Estrutura para armazenar resultados V2
full_results_v2 = {fs: {} for fs in FS_INDICES.keys()}

# Ratios de treino para V2
V2_RATIOS = [3, 5]

# Parametros fixos (Fase 1)
PARAMS_V2 = {
    'th_on': 0.5,
    'th_off': 0.3,
    'k': 3,
    'max_gap': 10.0,
    'min_duration': 5.0,
}

print("\n" + "="*60)
print("EXECUTANDO PIPELINE V2 (FASE 1 - RAPIDO)")
print("="*60)
print(f"\nTotal de experimentos: {len(FS_INDICES) * len(RUN_CFGS) * len(RUN_MODELS) * len(V2_RATIOS)}")
print("Iniciando loop V2...\n")

combos_v2 = list(itertools.product(
    FS_INDICES.keys(),
    RUN_CFGS,
    RUN_MODELS,
    V2_RATIOS
))

for fs_name, cfg_name, model_key, ratio_train in tqdm(combos_v2, desc='Experimentos V2', unit='exp'):
    cfg = WINDOW_CONFIGS[cfg_name]
    feat_index = FS_INDICES[fs_name].get(cfg_name, {})
    valid_subs = [s for s in VALID_SUBJECTS if s in feat_index]
    
    if not valid_subs:
        continue
    
    if cfg_name not in full_results_v2[fs_name]:
        full_results_v2[fs_name][cfg_name] = {}
    if model_key not in full_results_v2[fs_name][cfg_name]:
        full_results_v2[fs_name][cfg_name][model_key] = {}
    
    results_v2 = run_loso_v2(
        feat_index=feat_index,
        subjects=valid_subs,
        win_sec=cfg['win_sec'],
        overlap=cfg['overlap'],
        model_key=model_key,
        ratio_train=ratio_train,
        **PARAMS_V2
    )
    
    df_v2 = pd.DataFrame(results_v2)
    full_results_v2[fs_name][cfg_name][model_key][ratio_train] = (results_v2, df_v2)

print("\n✅ Todos os experimentos V2 concluídos!")


EXECUTANDO PIPELINE V2 (FASE 1 - RAPIDO)

Total de experimentos: 54
Iniciando loop V2...



Experimentos V2: 100%|██████████| 54/54 [1:29:39<00:00, 99.63s/exp] 


✅ Todos os experimentos V2 concluídos!


## 13.1. Resumo de métricas (V2)

Agrega as métricas por configuração após o loop V2 para comparar rapidamente as combinações.

In [20]:
# Resumo agregado por janela (V2)
metrics_window = [
    'accuracy', 'f1', 'sensitivity', 'specificity', 'precision',
    'auc_pr', 'far_per_hour', 'event_sensitivity', 'event_precision', 'event_far_per_hour'
 ]
param_cols = ['ratio', 'th_on', 'th_off', 'k', 'max_gap', 'min_duration']
v2_rows = []

for fs_name in FS_INDICES.keys():
    for cfg_name in RUN_CFGS:
        for model_key in RUN_MODELS:
            for ratio_train in V2_RATIOS:
                try:
                    _, df_v2 = full_results_v2[fs_name][cfg_name][model_key][ratio_train]
                    grouped = df_v2.groupby(param_cols, dropna=False)
                    for params, g in grouped:
                        row = {
                            'FS': fs_name,
                            'Janela': cfg_name,
                            'Modelo': model_key.upper(),
                        }
                        for col, val in zip(param_cols, params):
                            row[col] = val
                        for m in metrics_window:
                            row[m] = g[m].mean() if m in g else 0.0
                        v2_rows.append(row)
                except KeyError:
                    pass

df_v2_window_summary = pd.DataFrame(v2_rows)
print(f"Resumo V2 (janela) gerado: {len(df_v2_window_summary)} linhas")

if not df_v2_window_summary.empty:
    df_v2_window_summary = df_v2_window_summary.sort_values(
        by=['sensitivity', 'far_per_hour', 'auc_pr'],
        ascending=[False, True, False]
).reset_index(drop=True)
    display(df_v2_window_summary.head(10))

Resumo V2 (janela) gerado: 54 linhas


,FS,Janela,Modelo,ratio,th_on,th_off,k,max_gap,min_duration,accuracy,f1,sensitivity,specificity,precision,auc_pr,far_per_hour,event_sensitivity,event_precision,event_far_per_hour
0,FS-B,4s_50,XGB,3,0.5,0.3,3,10.0,5.0,0.913978,0.089511,0.663455,0.914926,0.051299,0.145791,152.305401,0.843056,0.024999,12.523163
1,FS-A,6s_50,RF,3,0.5,0.3,3,10.0,5.0,0.912670,0.095131,0.653580,0.913659,0.056040,0.160055,103.053330,0.848611,0.032994,9.748436
2,FS-B,5s_50,RF,3,0.5,0.3,3,10.0,5.0,0.903184,0.083270,0.652558,0.904178,0.047691,0.185342,137.227525,0.843056,0.032022,11.361762
3,FS-B,6s_50,XGB,3,0.5,0.3,3,10.0,5.0,0.913404,0.092978,0.647071,0.914448,0.052556,0.172293,102.084291,0.843056,0.032731,9.648932
4,FS-B,5s_50,XGB,3,0.5,0.3,3,10.0,5.0,0.905085,0.084881,0.637675,0.906161,0.048845,0.179914,134.381571,0.848611,0.030226,11.374241
5,FS-A,4s_50,XGB,3,0.5,0.3,3,10.0,5.0,0.914881,0.098117,0.633284,0.916064,0.059226,0.119530,150.315068,0.843056,0.024029,12.253852
6,FS-A,5s_50,RF,3,0.5,0.3,3,10.0,5.0,0.911020,0.086509,0.629266,0.912153,0.050543,0.148714,125.831907,0.843056,0.033029,10.974820
7,FS-B,4s_50,RF,3,0.5,0.3,3,10.0,5.0,0.906080,0.075950,0.623444,0.907107,0.042324,0.170350,166.297641,0.869444,0.025550,13.305261
8,FS-B,6s_50,RF,3,0.5,0.3,3,10.0,5.0,0.905485,0.090689,0.611577,0.906603,0.053236,0.194477,111.442390,0.843056,0.036961,10.320369
9,FS-A,5s_50,XGB,3,0.5,0.3,3,10.0,5.0,0.911839,0.101373,0.609613,0.913108,0.064142,0.127555,124.492762,0.837500,0.036812,10.863137


## 14. Salvar resultados V2

Consolida todas as execuções da V2 em CSV para análise e comparação externa.

In [21]:
# Consolidar tudo em um unico CSV
all_rows_v2 = []

for fs_name in FS_INDICES.keys():
    for cfg_name in RUN_CFGS:
        for model_key in RUN_MODELS:
            for ratio_train in V2_RATIOS:
                try:
                    results_v2, df_v2 = full_results_v2[fs_name][cfg_name][model_key][ratio_train]
                    for _, row in df_v2.iterrows():
                        row_dict = row.to_dict()
                        row_dict['fs_name'] = fs_name
                        row_dict['cfg_name'] = cfg_name
                        row_dict['model'] = model_key
                        all_rows_v2.append(row_dict)
                except KeyError:
                    pass

df_all_v2 = pd.DataFrame(all_rows_v2)
output_path_v2 = os.path.join(DATA_DIR, 'results_v2.csv')
df_all_v2.to_csv(output_path_v2, index=False)

print(f"✅ Resultados V2 salvos em: {output_path_v2}")
print(f"   Total de linhas: {len(df_all_v2)}")

✅ Resultados V2 salvos em: ./data\results_v2.csv
   Total de linhas: 648


## 15. Ranking V2

Ordena as configurações pela média por sujeito, priorizando **AUC-PR**, **sensibilidade** e **FAR**.

In [22]:
# Agregar por configuracao (media entre sujeitos)
metrics_v2 = [
    'accuracy', 'f1', 'sensitivity', 'specificity', 'precision',
    'auc_pr', 'far_per_hour', 'event_sensitivity', 'event_precision', 'event_far_per_hour'
 ]
param_cols = ['ratio', 'th_on', 'th_off', 'k', 'max_gap', 'min_duration']
ranking_rows_v2 = []

for fs_name in FS_INDICES.keys():
    for cfg_name in RUN_CFGS:
        for model_key in RUN_MODELS:
            for ratio_train in V2_RATIOS:
                try:
                    _, df_v2 = full_results_v2[fs_name][cfg_name][model_key][ratio_train]
                    grouped = df_v2.groupby(param_cols, dropna=False)
                    for params, g in grouped:
                        row = {
                            'FS': fs_name,
                            'Janela': cfg_name,
                            'Modelo': model_key.upper(),
                        }
                        for col, val in zip(param_cols, params):
                            row[col] = val
                        for m in metrics_v2:
                            row[m] = g[m].mean()
                        ranking_rows_v2.append(row)
                except KeyError:
                    pass

df_ranking_v2 = pd.DataFrame(ranking_rows_v2)
df_ranking_v2 = df_ranking_v2.sort_values(
    by=['auc_pr', 'sensitivity', 'far_per_hour'],
    ascending=[False, False, True]
).reset_index(drop=True)

ranking_path_v2 = os.path.join(DATA_DIR, 'ranking_v2.csv')
df_ranking_v2.to_csv(ranking_path_v2, index=False)

print("✅ Ranking V2 salvo")
print("\nTop 10 configuracoes (V2):")
display(df_ranking_v2.head(10))

✅ Ranking V2 salvo

Top 10 configuracoes (V2):


,FS,Janela,Modelo,ratio,th_on,th_off,k,max_gap,min_duration,accuracy,f1,sensitivity,specificity,precision,auc_pr,far_per_hour,event_sensitivity,event_precision,event_far_per_hour
0,FS-B,6s_50,RF,3,0.5,0.3,3,10.0,5.0,0.905485,0.090689,0.611577,0.906603,0.053236,0.194477,111.442390,0.843056,0.036961,10.320369
1,FS-B,6s_50,SVM,5,0.5,0.3,3,10.0,5.0,0.956355,0.158179,0.451002,0.958534,0.114932,0.186152,49.498280,0.670833,0.060313,5.436577
2,FS-B,5s_50,RF,3,0.5,0.3,3,10.0,5.0,0.903184,0.083270,0.652558,0.904178,0.047691,0.185342,137.227525,0.843056,0.032022,11.361762
3,FS-B,6s_50,RF,5,0.5,0.3,3,10.0,5.0,0.936183,0.129211,0.556542,0.937828,0.084573,0.185139,74.193397,0.795833,0.046034,7.754522
4,FS-B,5s_50,XGB,3,0.5,0.3,3,10.0,5.0,0.905085,0.084881,0.637675,0.906161,0.048845,0.179914,134.381571,0.848611,0.030226,11.374241
5,FS-B,6s_50,XGB,5,0.5,0.3,3,10.0,5.0,0.941815,0.139343,0.604193,0.943205,0.088865,0.178767,67.778702,0.843056,0.047729,7.224264
6,FS-A,6s_50,RF,5,0.5,0.3,3,10.0,5.0,0.947755,0.158470,0.566442,0.949371,0.114334,0.175071,60.415234,0.837500,0.064914,6.888710
7,FS-B,4s_50,RF,5,0.5,0.3,3,10.0,5.0,0.935683,0.109553,0.574784,0.937180,0.067103,0.173689,112.464254,0.843056,0.032057,10.222271
8,FS-B,6s_50,XGB,3,0.5,0.3,3,10.0,5.0,0.913404,0.092978,0.647071,0.914448,0.052556,0.172293,102.084291,0.843056,0.032731,9.648932
9,FS-B,6s_50,SVM,3,0.5,0.3,3,10.0,5.0,0.933430,0.097586,0.506008,0.935214,0.059315,0.171931,77.324218,0.697222,0.041963,7.410652


## 16. Comparação baseline vs V2

Compara as melhores configurações de cada combinação de FS × janela × modelo × ratio, e calcula ganhos/reduções por métrica.

In [24]:
print("\n" + "="*80)
print("COMPARACAO: BASELINE vs V2 (JANELA)")
print("="*80)

common_metrics = [
    'accuracy', 'f1', 'sensitivity', 'specificity', 'precision', 'auc_pr',
    'far_per_hour', 'event_sensitivity', 'event_precision', 'event_far_per_hour'
 ]

df_baseline_best = df_ranking_baseline.sort_values(
    by=['sensitivity', 'far_per_hour', 'auc_pr'],
    ascending=[False, True, False]
).groupby(['FS', 'Janela', 'Modelo', 'Ratio'], as_index=False).first()

df_v2_best = df_ranking_v2.sort_values(
    by=['auc_pr', 'sensitivity', 'far_per_hour'],
    ascending=[False, False, True]
).groupby(['FS', 'Janela', 'Modelo', 'ratio'], as_index=False).first()

# Normalizar ratio para o mesmo tipo antes do merge
df_baseline_best = df_baseline_best.copy()
df_v2_best = df_v2_best.copy()
df_baseline_best['ratio_int'] = df_baseline_best['Ratio'].astype(str).str.replace('1:', '', regex=False).astype(int)
df_v2_best['ratio_int'] = df_v2_best['ratio'].astype(int)

df_compare = df_baseline_best.merge(
    df_v2_best,
    left_on=['FS', 'Janela', 'Modelo', 'ratio_int'],
    right_on=['FS', 'Janela', 'Modelo', 'ratio_int'],
    suffixes=('_baseline', '_v2'),
    how='inner'
 )

comparison_rows = []
for _, row in df_compare.iterrows():
    out = row.to_dict()
    for m in common_metrics:
        baseline_val = row.get(f"{m}_baseline", 0)
        v2_val = row.get(f"{m}_v2", 0)
        if m in ['far_per_hour', 'event_far_per_hour'] and baseline_val > 0:
            out[f"{m}_improvement_%"] = ((baseline_val - v2_val) / baseline_val * 100)
        else:
            out[f"{m}_improvement_%"] = ((v2_val - baseline_val) / baseline_val * 100 if baseline_val > 0 else 0)
    comparison_rows.append(out)

df_comparison = pd.DataFrame(comparison_rows)
comparison_path = os.path.join(DATA_DIR, 'comparison_baseline_vs_v2.csv')
df_comparison.to_csv(comparison_path, index=False)

print(f"\n✅ Comparacao salva em: {comparison_path}")

print("\n" + "="*80)
print("ESTATISTICAS GERAIS DE MELHORIA")
print("="*80)

for m in common_metrics:
    improvement_col = f"{m}_improvement_%"
    if improvement_col in df_comparison:
        avg_improvement = df_comparison[improvement_col].mean()
        metric_label = m.replace('_', ' ').title()
        if m in ['far_per_hour', 'event_far_per_hour']:
            print(f"\n{metric_label}: {avg_improvement:+.1f}% (reducao media)")
        else:
            print(f"\n{metric_label}: {avg_improvement:+.1f}% (melhoria media)")

print("\n" + "="*80)
print("MELHOR CONFIGURACAO (V2)")
print("="*80)

best_idx = df_ranking_v2.iloc[0]
print(f"\nFS: {best_idx['FS']}")
print(f"Janela: {best_idx['Janela']}")
print(f"Modelo: {best_idx['Modelo']}")
print(f"\nParametros:")
print(f"  ratio: 1:{best_idx['ratio']}")
print(f"  th_on: {best_idx['th_on']}")
print(f"  th_off: {best_idx['th_off']}")
print(f"  k: {best_idx['k']}")
print(f"  max_gap: {best_idx['max_gap']}")
print(f"  min_duration: {best_idx['min_duration']}")
print(f"\nMetricas:")
print(f"  Accuracy: {best_idx['accuracy']:.3f}")
print(f"  F1: {best_idx['f1']:.3f}")
print(f"  Sensitivity: {best_idx['sensitivity']:.3f}")
print(f"  Precision: {best_idx['precision']:.3f}")
print(f"  AUC-PR: {best_idx['auc_pr']:.3f}")
print(f"  FAR: {best_idx['far_per_hour']:.2f} alarmes/hora")
print(f"  Event Sensitivity: {best_idx['event_sensitivity']:.3f}")
print(f"  Event FAR: {best_idx['event_far_per_hour']:.2f} alarmes/hora")

print("\n" + "="*80)
print("✅ COMPARACAO CONCLUIDA!")
print("="*80)


COMPARACAO: BASELINE vs V2 (JANELA)

✅ Comparacao salva em: ./data\comparison_baseline_vs_v2.csv

ESTATISTICAS GERAIS DE MELHORIA

Accuracy: +0.3% (melhoria media)

F1: +43.1% (melhoria media)

Sensitivity: +6.4% (melhoria media)

Specificity: +0.3% (melhoria media)

Precision: +56.5% (melhoria media)

Auc Pr: -2.4% (melhoria media)

Far Per Hour: +5.0% (reducao media)

Event Sensitivity: -5.0% (melhoria media)

Event Precision: +138.7% (melhoria media)

Event Far Per Hour: +76.5% (reducao media)

MELHOR CONFIGURACAO (V2)

FS: FS-B
Janela: 6s_50
Modelo: RF

Parametros:
  ratio: 1:3
  th_on: 0.5
  th_off: 0.3
  k: 3
  max_gap: 10.0
  min_duration: 5.0

Metricas:
  Accuracy: 0.905
  F1: 0.091
  Sensitivity: 0.612
  Precision: 0.053
  AUC-PR: 0.194
  FAR: 111.44 alarmes/hora
  Event Sensitivity: 0.843
  Event FAR: 10.32 alarmes/hora

✅ COMPARACAO CONCLUIDA!


## 18. Sumário final

Exibe métricas médias gerais, top 3 configurações do baseline e V2, e lista os arquivos salvos.

In [26]:
print("\n" + "="*80)
print("SUMARIO FINAL - BASELINE vs V2 (JANELA)")
print("="*80)

common_metrics = [
    'accuracy', 'f1', 'sensitivity', 'specificity', 'precision', 'auc_pr',
    'far_per_hour', 'event_sensitivity', 'event_precision', 'event_far_per_hour'
 ]

print("\n1. METRICAS GERAIS (media de todas as configuracoes)")
print("-" * 80)

for m in common_metrics:
    baseline_col = f"{m}_baseline"
    v2_col = f"{m}_v2"
    if baseline_col in df_comparison and v2_col in df_comparison:
        baseline_avg = df_comparison[baseline_col].mean()
        v2_avg = df_comparison[v2_col].mean()
        if m in ['far_per_hour', 'event_far_per_hour']:
            change = baseline_avg - v2_avg
            pct = (change / baseline_avg * 100) if baseline_avg > 0 else 0
            print(f"\n{m.upper()}:")
            print(f"  Baseline: {baseline_avg:.2f}")
            print(f"  V2: {v2_avg:.2f}")
            print(f"  Reducao: {change:.2f} ({pct:.1f}%)")
        else:
            change = v2_avg - baseline_avg
            pct = (change / baseline_avg * 100) if baseline_avg > 0 else 0
            print(f"\n{m.upper()}:")
            print(f"  Baseline: {baseline_avg:.3f}")
            print(f"  V2: {v2_avg:.3f}")
            print(f"  Melhoria: {change:+.3f} ({pct:+.1f}%)")

print("\n\n2. TOP 3 CONFIGURACOES - BASELINE")
print("-" * 80)
display(df_ranking_baseline.head(3)[
    ['FS', 'Janela', 'Ratio', 'Modelo', 'accuracy', 'sensitivity', 'far_per_hour', 'event_sensitivity', 'event_far_per_hour']
 ])

print("\n2. TOP 3 CONFIGURACOES - V2")
print("-" * 80)
display(df_ranking_v2.head(3)[
    ['FS', 'Janela', 'Modelo', 'ratio', 'th_on', 'th_off', 'k', 'max_gap', 'min_duration',
     'accuracy', 'sensitivity', 'far_per_hour', 'event_sensitivity', 'event_far_per_hour']
 ])

print("\n\n3. ARQUIVOS SALVOS")
print("-" * 80)
print("  • results_baseline.csv")
print("  • ranking_baseline.csv")
print("  • results_v2.csv")
print("  • ranking_v2.csv")
print("  • comparison_baseline_vs_v2.csv")

print("\n" + "="*80)
print("✅ PIPELINE V2 CONCLUIDO COM SUCESSO!")
print("="*80)


SUMARIO FINAL - BASELINE vs V2 (JANELA)

1. METRICAS GERAIS (media de todas as configuracoes)
--------------------------------------------------------------------------------

ACCURACY:
  Baseline: 0.928
  V2: 0.931
  Melhoria: +0.003 (+0.3%)

F1:
  Baseline: 0.074
  V2: 0.107
  Melhoria: +0.033 (+44.1%)

SENSITIVITY:
  Baseline: 0.526
  V2: 0.559
  Melhoria: +0.034 (+6.4%)

SPECIFICITY:
  Baseline: 0.930
  V2: 0.933
  Melhoria: +0.003 (+0.3%)

PRECISION:
  Baseline: 0.043
  V2: 0.069
  Melhoria: +0.026 (+59.4%)

AUC_PR:
  Baseline: 0.149
  V2: 0.145
  Melhoria: -0.004 (-2.9%)

FAR_PER_HOUR:
  Baseline: 104.18
  V2: 99.47
  Reducao: 4.70 (4.5%)

EVENT_SENSITIVITY:
  Baseline: 0.847
  V2: 0.805
  Melhoria: -0.042 (-5.0%)

EVENT_PRECISION:
  Baseline: 0.016
  V2: 0.037
  Melhoria: +0.022 (+137.6%)

EVENT_FAR_PER_HOUR:
  Baseline: 39.98
  V2: 9.28
  Reducao: 30.71 (76.8%)


2. TOP 3 CONFIGURACOES - BASELINE
--------------------------------------------------------------------------------


,FS,Janela,Ratio,Modelo,accuracy,sensitivity,far_per_hour,event_sensitivity,event_far_per_hour
0,FS-B,4s_50,1:3,RF,0.903247,0.609374,171.166518,0.854167,61.343936
1,FS-B,4s_50,1:3,XGB,0.905825,0.600211,166.424693,0.854167,61.559055
2,FS-A,4s_50,1:3,XGB,0.911284,0.577975,156.366943,0.854167,62.158336



2. TOP 3 CONFIGURACOES - V2
--------------------------------------------------------------------------------


,FS,Janela,Modelo,ratio,th_on,th_off,k,max_gap,min_duration,accuracy,sensitivity,far_per_hour,event_sensitivity,event_far_per_hour
0,FS-B,6s_50,RF,3,0.5,0.3,3,10.0,5.0,0.905485,0.611577,111.442390,0.843056,10.320369
1,FS-B,6s_50,SVM,5,0.5,0.3,3,10.0,5.0,0.956355,0.451002,49.498280,0.670833,5.436577
2,FS-B,5s_50,RF,3,0.5,0.3,3,10.0,5.0,0.903184,0.652558,137.227525,0.843056,11.361762




3. ARQUIVOS SALVOS
--------------------------------------------------------------------------------
  • results_baseline.csv
  • ranking_baseline.csv
  • results_v2.csv
  • ranking_v2.csv
  • comparison_baseline_vs_v2.csv

✅ PIPELINE V2 CONCLUIDO COM SUCESSO!


## 19. Fase 2 — tuning do modelo campeão (opcional)

Executa a busca de `th_on`, `th_off` e `k` **apenas** para a melhor configuração da Fase 1, reduzindo custo computacional.

In [27]:
# ============================================================
# FASE 2 - THRESHOLD TUNING APENAS NO MODELO CAMPEAO
# ============================================================
if df_ranking_v2.empty:
    print("⚠️ df_ranking_v2 está vazio. Execute a Fase 1 antes do tuning.")
else:
    champion = df_ranking_v2.iloc[0]
    fs_name = champion['FS']
    cfg_name = champion['Janela']
    model_name = champion['Modelo'].lower()
    ratio_train = int(champion['ratio']) if 'ratio' in champion else int(champion.get('Ratio', 3))
    
    cfg = WINDOW_CONFIGS[cfg_name]
    feat_index = FS_INDICES[fs_name].get(cfg_name, {})
    valid_subs = [s for s in VALID_SUBJECTS if s in feat_index]
    
    print("\n" + "="*70)
    print("FASE 2 - TUNING DO MODELO CAMPEAO")
    print("="*70)
    print(f"FS: {fs_name} | Janela: {cfg_name} | Modelo: {model_name.upper()} | Ratio: 1:{ratio_train}")
    
    # Grid de pos-processamento (Fase 2)
    th_on_list = [0.3, 0.4, 0.5, 0.6]
    th_off_list = [0.2, 0.3]
    k_list = [1, 3, 5]
    max_gap = 10.0
    min_duration = 5.0
    
    def _train_probs_loso(feat_index, subjects, win_sec, overlap, model_key, ratio_train):
        cache = []
        valid = [s for s in subjects if s in feat_index]
        for test_sub in valid:
            train_subs = [s for s in valid if s != test_sub]
            X_train_list, y_train_list = [], []
            for s in train_subs:
                fp, lp = feat_index[s]
                X_train_list.append(np.load(fp))
                y_train_list.append(np.load(lp))
            X_train = np.vstack(X_train_list)
            y_train = np.concatenate(y_train_list)
            X_train, y_train = apply_ratio_v2(X_train, y_train, ratio_train, win_sec, overlap)
            
            fp_test, lp_test = feat_index[test_sub]
            X_test = np.load(fp_test)
            y_test = np.load(lp_test)
            if len(X_train) == 0 or len(X_test) == 0:
                continue
            
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
            n_features = min(10, X_train.shape[1])
            selector = SelectKBest(mutual_info_classif, k=n_features)
            X_train = selector.fit_transform(X_train, y_train)
            X_test = selector.transform(X_test)
            
            if model_key == 'xgb':
                model = XGBClassifier(
                    n_estimators=100,
                    max_depth=5,
                    learning_rate=0.1,
                    random_state=RANDOM_SEED,
                    eval_metric='logloss'
                )
            elif model_key == 'svm':
                model = SVC(
                    kernel='rbf',
                    C=1.0,
                    gamma='scale',
                    probability=True,
                    random_state=RANDOM_SEED
                )
            elif model_key == 'rf':
                model = RandomForestClassifier(
                    n_estimators=100,
                    max_depth=10,
                    random_state=RANDOM_SEED
                )
            
            model.fit(X_train, y_train)
            probs = model.predict_proba(X_test)[:, 1]
            auc_pr = average_precision_score(y_test, probs) if len(np.unique(y_test)) > 1 else 0.0
            cache.append({
                'subject': test_sub,
                'y_test': y_test,
                'probs': probs,
                'auc_pr': auc_pr,
            })
        return cache
    
    cached_probs = _train_probs_loso(
        feat_index=feat_index,
        subjects=valid_subs,
        win_sec=cfg['win_sec'],
        overlap=cfg['overlap'],
        model_key=model_name,
        ratio_train=ratio_train
    )
    
    if not cached_probs:
        print("⚠️ Sem dados para tuning (cache vazio).")
    else:
        metrics_window = [
            'accuracy', 'f1', 'sensitivity', 'specificity', 'precision',
            'auc_pr', 'far_per_hour', 'event_sensitivity', 'event_precision', 'event_far_per_hour'
        ]
        tuning_rows = []
        
        for th_on, th_off, k in itertools.product(th_on_list, th_off_list, k_list):
            per_subject = []
            for item in cached_probs:
                y_test = item['y_test']
                probs = item['probs']
                detections_init = (probs >= th_on).astype(int)
                detections = apply_hysteresis_from_pred(probs, detections_init, th_on=th_on, th_off=th_off)
                detections = apply_k_consecutive(detections, k=k)
                events = probs_to_events(detections, cfg['win_sec'], cfg['overlap'])
                events = merge_events(events, max_gap=max_gap)
                events = filter_short_events(events, min_duration=min_duration)
                detections = events_to_detections(events, len(y_test), cfg['win_sec'], cfg['overlap'])
                
                metrics = compute_metrics(y_test, detections, cfg['win_sec'], cfg['overlap'])
                metrics.update(compute_event_metrics(y_test, detections, cfg['win_sec'], cfg['overlap']))
                metrics['auc_pr'] = item['auc_pr']
                per_subject.append(metrics)
            
            df_tmp = pd.DataFrame(per_subject)
            row = {
                'th_on': th_on,
                'th_off': th_off,
                'k': k,
                'max_gap': max_gap,
                'min_duration': min_duration,
            }
            for m in metrics_window:
                row[m] = df_tmp[m].mean() if m in df_tmp else 0.0
            tuning_rows.append(row)
        
        df_tuning = pd.DataFrame(tuning_rows)
        df_tuning = df_tuning.sort_values(
            by=['auc_pr', 'sensitivity', 'far_per_hour'],
            ascending=[False, False, True]
        ).reset_index(drop=True)
        
        print("\nTop 10 combinacoes (tuning do campeao):")
        display(df_tuning.head(10))


FASE 2 - TUNING DO MODELO CAMPEAO
FS: FS-B | Janela: 6s_50 | Modelo: RF | Ratio: 1:3

Top 10 combinacoes (tuning do campeao):


,th_on,th_off,k,max_gap,min_duration,accuracy,f1,sensitivity,specificity,precision,auc_pr,far_per_hour,event_sensitivity,event_precision,event_far_per_hour
0,0.3,0.2,1,10.0,5.0,0.803524,0.044212,0.735502,0.803457,0.023388,0.194477,234.598631,0.854167,0.012527,20.635794
1,0.3,0.3,1,10.0,5.0,0.818771,0.048358,0.723492,0.818855,0.025804,0.194477,216.203690,0.848611,0.014736,19.199680
2,0.3,0.2,3,10.0,5.0,0.852080,0.056567,0.714656,0.852379,0.030526,0.194477,176.195257,0.848611,0.019747,14.138172
3,0.4,0.2,1,10.0,5.0,0.838142,0.053261,0.709830,0.838380,0.028594,0.194477,192.886515,0.848611,0.017005,17.955543
4,0.4,0.2,3,10.0,5.0,0.869721,0.064980,0.699789,0.870187,0.035563,0.194477,154.930275,0.843056,0.024013,12.441023
5,0.3,0.2,5,10.0,5.0,0.888990,0.074844,0.693912,0.889621,0.041435,0.194477,131.739002,0.843056,0.036376,8.300437
6,0.4,0.3,1,10.0,5.0,0.851875,0.057831,0.687745,0.852329,0.031384,0.194477,176.225456,0.848611,0.019791,17.202032
7,0.4,0.2,5,10.0,5.0,0.899947,0.080291,0.677924,0.900723,0.044837,0.194477,118.484830,0.822222,0.037390,7.484475
8,0.5,0.2,1,10.0,5.0,0.867221,0.064915,0.670852,0.867859,0.035725,0.194477,157.684643,0.843056,0.021326,14.941319
9,0.3,0.3,3,10.0,5.0,0.881259,0.071043,0.669923,0.881876,0.039503,0.194477,140.962727,0.848611,0.025751,12.884595
